The datasets contains transactions made by credit cards in September 2013 by european cardholders. This dataset presents transactions that occurred in two days, where we have 492 frauds out of 245,807 transactions. The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.
It contains only numerical input variables which are the result of a PCA transformation. Unfortunately, due to confidentiality issues, we cannot provide the original features and more background information about the data.

Features V1, V2, ... V28 are the principal components obtained with PCA, the only features which have not been transformed with PCA are 'Time' and 'Amount.

Feature 'Time' contains the seconds elapsed between each transaction and the first transaction in the dataset.

The feature 'Amount' is the transaction Amount, this feature can be used for example-dependant cost-senstive learning.

Feature 'Class' is the response variable and it takes value 1 in case of fraud and 0 otherwise.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import seaborn as sns
import torch

In [2]:
df = pd.read_csv("creditcard.csv")

In [3]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [4]:
df["Time"].value_counts()

Time
163152.0    36
64947.0     26
68780.0     25
3767.0      21
3770.0      20
            ..
172784.0     1
172785.0     1
172786.0     1
172787.0     1
172792.0     1
Name: count, Length: 124592, dtype: int64

In [5]:
df["Amount"].describe()

count    284807.000000
mean         88.349619
std         250.120109
min           0.000000
25%           5.600000
50%          22.000000
75%          77.165000
max       25691.160000
Name: Amount, dtype: float64

In [6]:
df[df["Class"] == 1]["Amount"].sort_values(ascending=False)

176049    2125.87
6971      1809.68
249167    1504.93
89190     1402.16
81609     1389.56
           ...   
143334       0.00
69980        0.00
248296       0.00
93788        0.00
541          0.00
Name: Amount, Length: 492, dtype: float64

In [7]:
df.isna().sum().sum()

np.int64(0)

In [8]:
fraud = df[df["Class"] == 1]
non_fraud = df[df.Class == 0]

In [9]:
fraud

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
541,406.0,-2.312227,1.951992,-1.609851,3.997906,-0.522188,-1.426545,-2.537387,1.391657,-2.770089,...,0.517232,-0.035049,-0.465211,0.320198,0.044519,0.177840,0.261145,-0.143276,0.00,1
623,472.0,-3.043541,-3.157307,1.088463,2.288644,1.359805,-1.064823,0.325574,-0.067794,-0.270953,...,0.661696,0.435477,1.375966,-0.293803,0.279798,-0.145362,-0.252773,0.035764,529.00,1
4920,4462.0,-2.303350,1.759247,-0.359745,2.330243,-0.821628,-0.075788,0.562320,-0.399147,-0.238253,...,-0.294166,-0.932391,0.172726,-0.087330,-0.156114,-0.542628,0.039566,-0.153029,239.93,1
6108,6986.0,-4.397974,1.358367,-2.592844,2.679787,-1.128131,-1.706536,-3.496197,-0.248778,-0.247768,...,0.573574,0.176968,-0.436207,-0.053502,0.252405,-0.657488,-0.827136,0.849573,59.00,1
6329,7519.0,1.234235,3.019740,-4.304597,4.732795,3.624201,-1.357746,1.713445,-0.496358,-1.282858,...,-0.379068,-0.704181,-0.656805,-1.632653,1.488901,0.566797,-0.010016,0.146793,1.00,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279863,169142.0,-1.927883,1.125653,-4.518331,1.749293,-1.566487,-2.010494,-0.882850,0.697211,-2.064945,...,0.778584,-0.319189,0.639419,-0.294885,0.537503,0.788395,0.292680,0.147968,390.00,1
280143,169347.0,1.378559,1.289381,-5.004247,1.411850,0.442581,-1.326536,-1.413170,0.248525,-1.127396,...,0.370612,0.028234,-0.145640,-0.081049,0.521875,0.739467,0.389152,0.186637,0.76,1
280149,169351.0,-0.676143,1.126366,-2.213700,0.468308,-1.120541,-0.003346,-2.234739,1.210158,-0.652250,...,0.751826,0.834108,0.190944,0.032070,-0.739695,0.471111,0.385107,0.194361,77.89,1
281144,169966.0,-3.113832,0.585864,-5.399730,1.817092,-0.840618,-2.943548,-2.208002,1.058733,-1.632333,...,0.583276,-0.269209,-0.456108,-0.183659,-0.328168,0.606116,0.884876,-0.253700,245.00,1


In [10]:
df["Class"].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [11]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [12]:
y = df["Class"]
X = [x for x in df.columns.tolist() if x != "Class"]

In [13]:
X = df[X]

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
221940,142756.0,-1.366076,1.791214,-0.792605,0.952248,0.227136,-0.348030,0.095412,0.315024,-0.653622,...,0.412933,1.054074,-0.396830,-0.502498,-0.085598,0.253104,-1.009860,-0.179613,5.82,0
176474,122784.0,-2.515764,1.013633,0.698466,-1.499256,-2.229798,-0.273835,-1.462977,1.213240,0.281698,...,-0.123939,0.193094,0.045820,-0.041629,-0.588032,0.557284,-0.206537,0.028637,25.03,0
119441,75443.0,1.126376,-1.098248,-0.577764,-0.759379,-0.332117,0.416791,-0.313358,0.087788,-0.833831,...,-0.373034,-0.837598,-0.244608,-1.124420,0.397179,1.165232,-0.091532,-0.005090,148.91,0
11834,20318.0,1.180014,-0.593548,0.182817,-0.360502,-0.928133,-0.967612,-0.229465,-0.305657,-0.028377,...,-0.474115,-0.892734,-0.006933,0.501227,0.195598,0.932473,-0.125070,0.002708,102.45,0
59293,48780.0,1.068566,-0.052581,1.389764,1.783097,-0.761890,0.437976,-0.459556,0.202723,1.063006,...,-0.384900,-0.566345,0.116626,0.437554,0.421707,-0.533001,0.100001,0.033441,4.98,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175687,122444.0,1.958570,0.569780,-1.482448,3.505560,1.145545,0.406768,0.392095,-0.013680,-1.318480,...,0.275833,0.811284,-0.097958,0.287677,0.459237,0.269125,-0.075517,-0.072535,7.57,0
237879,149438.0,-1.159829,-5.496311,-2.064649,1.634687,-2.285164,0.154148,1.493324,-0.358193,0.810723,...,1.248269,0.102807,-1.265288,0.113170,-0.787521,-0.434512,-0.269436,0.232849,1609.65,0
109657,71479.0,-0.087460,-0.575980,1.419887,-2.108182,-1.307938,-0.651481,-1.639599,-1.804902,-1.870940,...,1.473243,-0.433978,-1.161909,0.452557,0.647263,-0.184979,0.312055,0.226985,111.00,0
246910,153386.0,0.172650,1.000114,-0.761956,-0.881716,1.460541,0.029992,0.884086,0.069439,-0.312853,...,-0.346501,-0.855980,-0.039408,-0.388828,-0.311863,0.150900,0.220624,0.065632,2.58,0


In [154]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.3, stratify=y
)

In [ ]:
def func_train_test_val_split(X, y):
    """
    splits into train,test,val
    @return:  X_train,y_train,X_val,y_val,X_test,y_test
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, random_state=42, test_size=0.3, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.1, random_state=42
    )
    return X_train, y_train, X_val, y_val, X_test, y_test

In [155]:
scaler = StandardScaler()

In [156]:
X_train_model = scaler.fit_transform(X_train)

In [157]:
X_test_model = scaler.transform(X_test)

In [158]:
X_train_model

array([[ 1.25799168, -0.00557826,  0.4278193 , ...,  0.57962275,
         0.25246159, -0.3210821 ],
       [ 0.93971289,  0.90410915, -0.11013829, ..., -0.1835134 ,
        -0.17372534,  0.24347542],
       [-0.52896001, -0.55054428, -2.6769007 , ..., -0.77644611,
         0.68944825,  4.79759381],
       ...,
       [-0.71876914, -0.76142256,  0.39801874, ..., -1.09887677,
         0.13544642, -0.2030889 ],
       [-1.2549473 ,  0.54477537,  0.04470792, ...,  0.07705819,
         0.07389814, -0.16977736],
       [-1.39085013, -0.30368651,  0.46865476, ...,  0.92950974,
         0.62433914, -0.32163531]], shape=(199364, 30))

## model building cnn torch

In [159]:
device = "mps"

In [160]:
X_train_model = torch.tensor(X_train_model, dtype=torch.float32, device=device)
X_test_model = torch.tensor(X_test_model, dtype=torch.float32, device=device)
y_train_model = y_train.to_numpy()
y_train_model = torch.tensor(y_train_model, dtype=torch.float32, device=device)
y_test_model = torch.tensor(y_test.to_numpy(), dtype=torch.float32, device=device)

In [187]:
X_train_model.shape

torch.Size([199364, 30])

In [195]:
X_train_model.unsqueeze(0)

torch.Size([1, 199364, 30])

In [161]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_model, y_train_model)
test_ds = TensorDataset(X_test_model, y_test_model)

In [162]:
BATCH_SIZE = 64
epochs = 300
torch.manual_seed(42)
train_dl = DataLoader(train_ds, BATCH_SIZE)
test_dl = DataLoader(test_ds, BATCH_SIZE)

In [163]:
x = next(iter(train_dl))[0]

In [165]:
x, x.dtype

(tensor([[ 1.2580e+00, -5.5783e-03,  4.2782e-01,  ...,  5.7962e-01,
           2.5246e-01, -3.2108e-01],
         [ 9.3971e-01,  9.0411e-01, -1.1014e-01,  ..., -1.8351e-01,
          -1.7373e-01,  2.4348e-01],
         [-5.2896e-01, -5.5054e-01, -2.6769e+00,  ..., -7.7645e-01,
           6.8945e-01,  4.7976e+00],
         ...,
         [ 1.4014e+00, -4.9629e+00, -5.9285e+00,  ...,  5.6809e+00,
          -3.1411e+00,  2.4631e+00],
         [ 1.3027e+00,  2.7173e-02,  2.4414e-01,  ..., -5.0126e-01,
          -7.0262e-01,  3.6740e-01],
         [-2.0384e-01,  3.6830e-01, -8.8350e-01,  ...,  3.2328e-02,
           2.0555e-01,  8.1645e-01]], device='mps:0'),
 torch.float32)

In [130]:
from torch import nn


class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=1, kernel_size=3, padding="same"),
            nn.ReLU(),
            nn.MaxPool1d(2, 2),
            nn.Dropout(0.2),
            nn.Flatten(),
            nn.Sigmoid(),
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)

In [166]:
import torch
import torch.nn as nn


class KaggleFraudCNN(nn.Module):
    def __init__(self):
        super(KaggleFraudCNN, self).__init__()

        # Conv1d expects input: [Batch_Size, Channels, Sequence_Length] -> [64, 1, 30]
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=30, kernel_size=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2)

        # FIX: Changed in_channels from 32 to 30 to match conv1's out_channels
        self.conv2 = nn.Conv1d(in_channels=30, out_channels=64, kernel_size=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        self.flatten = nn.Flatten()

        # FIX: Verified shape tracking. After pool2, sequence length is exactly 6.
        # [64, 64, 6] flattens out to 64 * 6 = 384 features.
        self.fc1 = nn.Linear(64 * 6, 64)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 1)
        # self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Cast to 32-bit float to prevent precision errors
        x = x.float()

        # 2. FIX: Check if input is a single unbatched sample [30]
        if x.dim() == 1:
            # Add a fake batch dimension -> [1, 30]
            x = x.unsqueeze(0)

        # 3. Safely flatten any accidental trailing dimensions -> [Batch, 30]
        x = x.flatten(1)

        # 4. Reshape dynamically into Conv1d format -> [Batch, 1, 30]
        x = x.view(x.size(0), 1, -1)

        # 5. First Conv block
        x = self.pool1(self.relu1(self.conv1(x)))

        # 6. Second Conv block
        x = self.pool2(self.relu2(self.conv2(x)))

        # 7. Flatten and Classify
        x = self.flatten(x)
        x = self.relu3(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [132]:
model = Model()

In [167]:
model = KaggleFraudCNN()

In [169]:
model = model.to(device)

In [202]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = torch.nn.BCEWithLogitsLoss()


device = "mps"

In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class TrainResult:
    train_loss: List[float]
    val_loss: List[float]
    train_acc: List[float]
    val_acc: List[float]

In [ ]:
from tqdm.auto import tqdm
from typing import List


def train(epochs=epochs):
    history = {
        "train_acc": List[float],
        "val_acc": List[float],
        "train_loss": [],
        "val": List[float],
    }
    bess_loss = float("inf")
    for i in range(epochs):
        for X_batch, y_batch in train_ds:
            # X_batch,y_batch = X_batch.to(device),y_batch.to(device)
            # X_batch=X_batch.to(device)
            y_pred = model(X_batch)
            # 1. Flatten both predictions and targets to a clean 1D array of shapes [Batch_Size]
            y_pred = y_pred.view(-1)
            y_batch = y_batch.view(-1)
            y_batch = y_batch.float()

            # y_batch=y_batch.to(device)

            # 2. Ensure your target labels match the 32-bit float type of the predictions

            # 3. Your loss function will now run perfectly
            loss = loss_fn(y_pred, y_batch)

            # print(y_batch,y_pred)
            # loss = loss_fn(y_pred,y_batch)
            history["train_loss"].append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"{i+1}/{epochs} loss={history['train_loss'][-1]}")
    return history

In [151]:
device = torch.device("mps" if torch.cuda.is_available() else "cpu")

In [220]:
from tqdm.auto import tqdm
import torch

history = TrainResult(train_loss=[], val_loss=[], train_acc=[], val_acc=[])


def train(epochs):
    # FIX: Initialize history with clean, empty Python lists []
    best_loss = float("inf")

    # Optional: Set up device configuration (GPU/CPU)

    for epoch in range(epochs):
        model.train()  # Set model to training mode (enables dropout)
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        # Wrapped with tqdm for an elegant progress bar
        for X_batch, y_batch in tqdm(train_ds, desc=f"Epoch {epoch+1}/{epochs}"):
            train_loss = 0
            # Send data to device
            # X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # Forward pass
            y_pred = model(X_batch).view(-1)

            # Align shapes and types
            # y_pred = y_pred.view(-1)
            y_batch = y_batch.view(-1).float()

            # Compute loss
            loss = loss_fn(y_pred, y_batch)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track batch metrics safely using .item() to prevent GPU memory leaks
            running_loss += loss.item() * X_batch.size(0)

            # Compute binary training accuracy
            # If your model ends with Sigmoid, threshold at 0.5.
            # If it uses Logits (BCEWithLogitsLoss), threshold at 0.0.
            # preds = (y_pred >= 0.5).float() if hasattr(model, 'sigmoid') else (y_pred >= 0.0).float()
            preds = (y_pred >= 0).float()

            # print(preds)
            correct_train += (preds == y_batch).sum().item()
            total_train += y_batch.size(0)

        # Calculate and record final epoch metrics
        epoch_loss = running_loss / total_train
        epoch_acc = correct_train / total_train
        history.train_loss.append(epoch_loss)
        history.train_acc.append(epoch_acc)
        print(epoch_loss)
        print(
            f"Epoch [{epoch+1}/{epochs}] Summary -> Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}"
        )
        print(best_loss)
        # Save best model checkpoint if loss improves
        if epoch_loss < best_loss:
            best_loss = epoch_loss

            torch.save(model.state_dict(), "best_model.pth")

    return history

In [227]:
train(1)

Epoch 1/1:   0%|          | 0/199364 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [186]:
for n, m in model.named_parameters():
    print(n)

conv1.weight
conv1.bias
conv2.weight
conv2.bias
fc1.weight
fc1.bias
fc2.weight
fc2.bias


In [223]:
@torch.no_grad()
def evaluate(epochs=epochs):
    model.eval()
    correct = 0
    total = 0
    incorrect = 0
    running_loss = 0
    for epoch in range(1, epochs + 1):
        for X_batch, y_batch in test_dl:
            y_pred = (model(X_batch).view(-1) >= 0).float()

            correct += (y_batch == y_pred).sum().item()
            total += y_batch.size(0)
            # correct_epoch = correct_eval/total_eval
            # loss = loss_fn(y_pred,y_batch)
            # # optimizer.zero_grad()
            # # loss.backward()
            # # optimizer.step()

            # running_loss += loss.item() * X_batch.size(0)
            incorrect = total - correct
        history.val_acc.append(correct / total)
        history.val_loss.append(incorrect / total)

In [224]:
evaluate(1)

In [226]:
history

TrainResult(train_loss=[], val_loss=[0.9982678510820079], train_acc=[], val_acc=[0.0017321489179921118])

In [ ]:
for epoch in